# SPACEMAP V4: ROI Extraction from Merged Images

This notebook extracts Regions of Interest (ROIs) from the merged CODEX and H&E images using GeoJSON boundary files. It processes each ROI as a separate pyramidal OME-TIFF file for downstream analysis.

## Prerequisites:
- **Completed Registration**: Run `Step1_registration.ipynb` first to generate merged images
- **Merged Images**: Available in `outputs/registration/merged/`
- **GeoJSON Boundaries**: ROI coordinates in `data/boundaries/`

## Key Features:
- **Memory-efficient processing**: Handles large images by processing one channel at a time
- **Pyramidal output**: Creates multi-resolution ROI images compatible with QuPath
- **Metadata preservation**: Maintains channel information and spatial calibration from merged images
- **Batch processing**: Processes all available merged images and corresponding ROI boundaries
- **ROI names**: Taken from the GeoJSON — QuPath's annotation `name` or its classification, or a `label` property; each ROI file is named after its ROI
- **Modular design**: Uses functions from `src/` modules for consistent processing

## Workflow Overview:
1. **Setup and Import** - Load libraries and utility functions from `src/` modules
2. **Configuration** - Define paths matching registration pipeline output structure
3. **Discovery** - Find merged images and corresponding GeoJSON boundary files
4. **ROI Extraction** - Process each ROI from each merged image with preserved metadata
5. **Validation** - Verify output quality and generate processing summary

## 1. Import Libraries and Setup

Import all necessary libraries and custom utility functions from the `src/` directory.

The modular design reuses functions from:
- **`utils.py`**: Common utilities like logging and GeoJSON handling
- **`registration.py`**: ROI processing functions aligned with registration pipeline

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import tifffile
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
project_root = Path('..').resolve()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

from spacemap.utils import (
    log_time, load_geojson, extract_bboxes_from_geojson,
    extract_rois_with_geometry_from_geojson, resolve_roi_label)
from spacemap.background_subtraction_utils import get_channel_info, get_image_mpp, normalize_image_stem, is_supported_image_file
from spacemap.registration import process_rois

print(" All libraries imported successfully!")
print(f" Project root: {project_root}")
print(f" Source path: {src_path}")

In [ ]:
import importlib
from spacemap import registration
from spacemap import utils
importlib.reload(utils)
importlib.reload(registration)
from spacemap.registration import process_rois
print("Modules reloaded successfully.")

## 2. Configure Directories and Parameters

Configure all input/output directories to align with the registration pipeline structure.

The ROI extraction uses:
- **Input**: Merged images from `outputs/registration/merged/` (Step1 output)
- **Output**: ROI files saved to `outputs/registration/roi/` (organized structure)
- **Boundaries**: GeoJSON files from `data/boundries/` (existing directory)

In [ ]:
# =============================================================================
# DIRECTORY CONFIGURATION FOR ROI EXTRACTION
# =============================================================================
log_time("Setting up directories and parameters for ROI extraction...")

geojson_dir = '../data/boundaries/'
merged_dir = '../outputs/registration/merged/'

rois_dir = '../outputs/registration/roi/'

os.makedirs(rois_dir, exist_ok=True)
print(f" Created/verified ROI output directory: {rois_dir}")

sample_input_mpp = 0.5083
if os.path.exists(merged_dir):
    sample_images = [f for f in os.listdir(merged_dir) if is_supported_image_file(f)]
    if sample_images:
        sample_input_mpp = float(get_image_mpp(os.path.join(merged_dir, sample_images[0]), default=sample_input_mpp).get('codex', sample_input_mpp))

mpp = {
    'codex': sample_input_mpp
}

roi_params = {
    'tile_size': (256, 256),
    'n_pyramid_levels': 2,
    'compression': 'deflate'
}

print(f"\n Directory Configuration:")
print(f"   GeoJSON boundaries: {geojson_dir}")
print(f"   Merged images (from spacemap.registration): {merged_dir}")
print(f"   ROI outputs: {rois_dir}")
print(f"\n Processing Parameters:")
print(f"   Pixel size: {mpp['codex']} μm/pixel")
print(f"   Tile size: {roi_params['tile_size']}")
print(f"   Pyramid levels: {roi_params['n_pyramid_levels']}")

required_dirs = [geojson_dir, merged_dir]
missing_dirs = [d for d in required_dirs if not os.path.exists(d)]

if missing_dirs:
    print(f"\n   Missing required directories:")
    for d in missing_dirs:
        print(f"   {d}")
    print("Please ensure the registration notebook (Step1) has been run successfully.")
else:
    print(f"\n  All required directories found!")

## 3. Discover Available Files

Scan for merged images from the registration pipeline and their corresponding GeoJSON boundary files.

The discovery process:
- **Merged Images**: Looks for `*_merged.ome.tiff` files (updated naming from Step1)
- **GeoJSON Files**: Matches each image to `*_roi_bounding_boxes_coords.geojson`
- **Validation**: Verifies ROI count and file integrity before processing

In [ ]:
# =============================================================================
# DISCOVER IMAGES AND GEOJSON FILES
# =============================================================================
log_time("Scanning for images and GeoJSON boundary files...")

image_files = []
if os.path.exists(merged_dir):
    image_files = [f for f in os.listdir(merged_dir) if is_supported_image_file(f)]
else:
    print(f"Image directory not found: {merged_dir}")

print(f"Found {len(image_files)} images:")
for i, img_file in enumerate(image_files, 1):
    print(f"   {i}. {img_file}")

# The ROI Extraction tab matches either naming; process_rois() now does the same.
def _geojson_candidates(base_name):
    return [f"{base_name}_roi_bounding_boxes_coords.geojson", f"{base_name}.geojson"]

valid_pairs = []
for img_file in image_files:
    base_name = normalize_image_stem(img_file)
    if base_name.endswith('_merged'):
        base_name = base_name[:-7]

    match = next((n for n in _geojson_candidates(base_name)
                  if os.path.exists(os.path.join(geojson_dir, n))), None)

    if match:
        valid_pairs.append((img_file, match))
        geojson_data = load_geojson(os.path.join(geojson_dir, match))
        rois = extract_rois_with_geometry_from_geojson(geojson_data) if geojson_data else []
        names = ', '.join(label for _bbox, label, _geom in rois) or 'none'
        print(f"   {img_file}  ->  {match}  ({len(rois)} ROIs: {names})")
    else:
        print(f"   {img_file}  ->  no GeoJSON (looked for {' or '.join(_geojson_candidates(base_name))})")

print(f"\nImages: {len(image_files)}   Complete pairs: {len(valid_pairs)}")


In [ ]:
# =============================================================================
# CONFIGURE CHANNELS (OPTIONAL)
# =============================================================================
# Define which channels to keep, rename, or recolor.
# Leave channels_to_keep as None to keep all channels.

# 1. Filter Channels
# channels_to_keep = [
#     'DAPI',
#     'CD4',
#     'CD8',
#     # ... add more
# ]
channels_to_keep = None  # Set to None to keep all

# 2. Rename Channels (Optional)
# Same as the Name column in the ROI Extraction tab's marker list.
# Format: {'Old Name': 'New Name'}
rename_channels = {
    # 'Old_Name': 'New_Name',
}

# 3. Recolor Channels (Optional)
# Keys use the NEW name when a channel is also renamed above.
# Format: {'Channel Name': (R, G, B)} or '#RRGGBB'
channel_colors = {
    # 'DAPI': '#0000FF',
}

# Create configuration object
channel_config = {
    'keep': channels_to_keep,
    'rename': rename_channels,
    'colors': channel_colors
}

print("Channel configuration set.")
if channels_to_keep:
    print(f"Keeping {len(channels_to_keep)} channels.")
else:
    print("Keeping all channels.")

In [ ]:
# =============================================================================
# DISCOVER CHANNELS
# =============================================================================
from spacemap.background_subtraction_utils import get_channel_info

if image_files:
    # Pick the first image to show channels
    sample_img = os.path.join(merged_dir, image_files[0])
    print(f"Analyzing channels from: {image_files[0]}")
    
    try:
        c_names, c_colors, c_meta, _ = get_channel_info(sample_img)
        
        print(f"\nFound {len(c_names)} channels:")
        print("-" * 60)
        print(f"{'Index':<6} {'Channel Name':<30} {'Color':<10}")
        print("-" * 60)
        
        for i, (name, color) in enumerate(zip(c_names, c_colors)):
            print(f"{i:<6} {name:<30} {color:<10}")
            
        # Generate a list for copy-pasting
        print("\nCopy this list to the next cell to configure channels:")
        print("channels_to_keep = [")
        for name in c_names:
            print(f"    '{name}',")
        print("]")
        
    except Exception as e:
        print(f"Error reading channels: {e}")
else:
    print("No images found to analyze channels.")

## 4. Extract ROIs from Merged Images

Process each ROI using the modular functions from `src/registration.py`.

The extraction process:
- **Memory Efficient**: Processes large images channel by channel
- **Metadata Preservation**: Maintains all channel information and spatial calibration
- **Quality Control**: Validates each ROI extraction and reports success/failure

In [ ]:
# =============================================================================
# PROCESS ROI EXTRACTION
# =============================================================================
if valid_pairs:
    log_time("Starting ROI extraction process...")
    
    try:
        # Use our utility function to process all ROIs
        process_rois(
            merged_dir=merged_dir,
            geojson_dir=geojson_dir, 
            rois_dir=rois_dir,
            mpp=mpp,
            channel_config=channel_config
        )
        
        log_time(" ROI extraction process completed!")
        
    except Exception as e:
        log_time(f" Error during ROI extraction: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("  No valid image pairs available for ROI extraction")

## 5. Validate Results and Summary

Analyze the extracted ROI files and generate processing statistics.

The validation includes:
- **File Count Verification**: Confirms all expected ROIs were extracted
- **Size Analysis**: Reports storage requirements for downstream analysis
- **Quality Check**: Visual inspection of sample ROIs to verify extraction success

In [ ]:
# =============================================================================
# VALIDATE RESULTS AND GENERATE SUMMARY
# =============================================================================
log_time("Validating ROI extraction results...")

# Scan the ROI output directory
roi_files = []
if os.path.exists(rois_dir):
    roi_files = [f for f in os.listdir(rois_dir) if f.endswith('.ome.tiff')]

print(f" ROI Extraction Results:")
print(f"   Total ROI files generated: {len(roi_files)}")

if roi_files:
    # Group ROIs by base name (heuristic: split by last underscore if possible, or just list all)
    # Since naming is flexible ({stem}_{label}.ome.tiff), grouping is harder without knowing stems.
    # We'll just list them.
    
    print(f"   Generated Files:")
    
    total_size_gb = 0
    for roi_file in sorted(roi_files):
        roi_path = os.path.join(rois_dir, roi_file)
        file_size = os.path.getsize(roi_path)
        file_size_mb = file_size / (1024**2)
        total_size_gb += file_size / (1024**3)
        
        print(f"    - {roi_file} ({file_size_mb:.2f} MB)")
    
    print(f"\n Total Output Size: {total_size_gb:.2f} GB")
    
    # Sample a few ROIs for quality check visualization
    print(f"\n Quality Check - Displaying sample ROIs:")
    
    sample_rois = sorted(roi_files)[:6]  # Show first 6 ROIs
    if sample_rois:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, roi_file in enumerate(sample_rois):
            roi_path = os.path.join(rois_dir, roi_file)
            
            try:
                with tifffile.TiffFile(roi_path) as tif:
                    # Display first channel
                    first_channel = tif.pages[0].asarray()
                    n_channels = len(tif.pages)
                    
                    # Try to get channel name from metadata
                    try:
                        desc = tif.pages[0].description
                        root = ET.fromstring(desc)
                        ns = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
                        ch_elem = root.find('.//ome:Channel', ns)
                        ch_name = ch_elem.attrib.get('Name', 'Unknown') if ch_elem is not None else 'Unknown'
                    except:
                        ch_name = "Channel 1"

                    ax = axes[i]
                    im = ax.imshow(first_channel, cmap='gray', vmax=np.percentile(first_channel, 99))
                    ax.set_title(f'{roi_file}\n{n_channels} channels\n{ch_name}', fontsize=8)
                    ax.axis('off')
                    
                    # Add colorbar
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                    
            except Exception as e:
                axes[i].text(0.5, 0.5, f'Error loading\\n{roi_file}\\n{str(e)}', 
                           ha='center', va='center', transform=axes[i].transAxes)
                axes[i].axis('off')
        
        # Hide unused subplots
        for i in range(len(sample_rois), 6):
            axes[i].set_visible(False)
        
        plt.suptitle('Sample ROI Quality Check', fontsize=16)
        plt.tight_layout()
        plt.show()

else:
    print("    No ROI files found in output directory")
    print("   Please check the processing logs for errors")

print(f"\n  ROI extraction validation completed!")
print(f"  All ROI files saved in: {rois_dir}")